# DeepLabV3-ResNet50 Transfer Learning: CHASEDB1 → DRIVE

**Task:** Binary retinal vessel segmentation.

This notebook follows the same training style as `02_SegFormer_B0_DRIVE.ipynb`, but replaces SegFormer-B0 with **DeepLabV3-ResNet50**.

Pipeline:
1. Clone GitHub repo and set working directory.
2. Set random seed and shared training configuration.
3. Load processed CHASEDB1 as the **source dataset**.
4. Load processed DRIVE as the **target / transfer dataset**.
5. Build **DeepLabV3-ResNet50** for binary segmentation.
6. Pretrain on CHASEDB1.
7. Load CHASEDB1 checkpoint and fine-tune on DRIVE.
8. Evaluate on DRIVE validation/test sets.
9. Visualize segmentation results as **Image | Ground Truth | Prediction**.

Common parameters such as `SEED`, `IMG_SIZE`, `BATCH_SIZE`, `NUM_EPOCHS`, `LR`, `WEIGHT_DECAY`, `PATIENCE`, `BEST_METRIC`, and `AMP` follow the SegFormer notebook style. Dataset-specific parameters are separated for CHASEDB1 and DRIVE so they can be tuned independently.

In [ ]:
!pip -q install albumentations opencv-python tqdm pandas matplotlib

In [ ]:
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/DannyPhongcoderso1/DL-retinal-vessel-segmentation-final.git"
REPO_DIR = Path("/kaggle/working/DL-retinal-vessel-segmentation-final")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Using repo:", REPO_DIR)
print("CWD:", Path.cwd())

In [ ]:
# Ensure the repository root (containing 'src/') is on sys.path for imports
import sys
from pathlib import Path
print('Notebook CWD:', Path.cwd())
repo_root = None
for p in [Path.cwd()] + list(Path.cwd().parents):
    if (p / 'src').exists():
        repo_root = p
        break
possible = Path('/kaggle/working/DL-retinal-vessel-segmentation-final')
if repo_root is None and (possible / 'src').exists():
    repo_root = possible
if repo_root is None:
    print('WARNING: Could not find repo root containing src/. Set REPO_DIR manually or adjust path.')
else:
    repo_root = repo_root.resolve()
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    print('Added to sys.path:', repo_root)

In [ ]:
import os
from pathlib import Path
import random
import zipfile

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import cv2

from torchvision.models.segmentation import deeplabv3_resnet50


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 1. Shared configuration

These parameters are shared across CHASEDB1 source training and DRIVE transfer training. You can tune the dataset-specific blocks below without changing the whole notebook.

In [ ]:
# ============================================================
# COMMON TRAINING CONFIGURATION
# Shared by CHASEDB1 and DRIVE
# ============================================================

SEED = 42
set_seed(SEED)

WORK_DIR = Path("/kaggle/working")

IMG_SIZE = 512
BATCH_SIZE = 4
NUM_WORKERS = 2
PIN_MEMORY = True

# Common optimizer/training defaults.
# Dataset-specific cells below may override LR / NUM_EPOCHS / PATIENCE.
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 50
PATIENCE = 10
BEST_METRIC = "dice"  # "dice" or "iou"
THRESHOLD = 0.5
AMP = torch.cuda.is_available()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("AMP:", AMP)

## 2. Dataset configuration

This notebook expects processed datasets in the same `.pt` format used by the SegFormer notebook:

```text
DATA_ROOT/
  train/*.pt
  val/*.pt
  test/*.pt
```

Each `.pt` sample should contain `image` and one of `manual`, `manual_2`, or `mask`. Update the Kaggle dataset paths/slugs below to match your uploaded datasets.

In [ ]:
# ============================================================
# SOURCE DATASET: CHASEDB1
# ============================================================

CHASE_DATA_DIR = Path("/kaggle/input/datasets/ngchnhphong/dl-src-transfer-dataset")  # update if needed
CHASE_ZIP_PATH = CHASE_DATA_DIR / "CHASE_processed_dataset.zip"                       # update filename if needed
CHASE_EXTRACT_DIR = WORK_DIR / "CHASE_processed_dataset"

CHASE_IMG_SIZE = IMG_SIZE
CHASE_BATCH_SIZE = BATCH_SIZE
CHASE_NUM_EPOCHS = 50
CHASE_LR = 1e-4
CHASE_WEIGHT_DECAY = WEIGHT_DECAY
CHASE_PATIENCE = 10
CHASE_BEST_MODEL_PATH = WORK_DIR / "best_deeplabv3_resnet50_chase.pth"
CHASE_HISTORY_PATH = WORK_DIR / "deeplabv3_resnet50_chase_history.csv"
CHASE_RESULTS_PATH = WORK_DIR / "deeplabv3_resnet50_chase_results.csv"

# ============================================================
# TARGET DATASET: DRIVE
# ============================================================

DRIVE_DATA_DIR = Path("/kaggle/input/datasets/ngchnhphong/dl-src-transfer-dataset")
DRIVE_ZIP_PATH = DRIVE_DATA_DIR / "DRIVE_processed_dataset.zip"
DRIVE_EXTRACT_DIR = WORK_DIR / "DRIVE_processed_dataset"

DRIVE_IMG_SIZE = IMG_SIZE
DRIVE_BATCH_SIZE = BATCH_SIZE
DRIVE_NUM_EPOCHS = 50
DRIVE_LR = 5e-5
DRIVE_WEIGHT_DECAY = WEIGHT_DECAY
DRIVE_PATIENCE = 10
DRIVE_BEST_MODEL_PATH = WORK_DIR / "best_deeplabv3_resnet50_drive.pth"
DRIVE_HISTORY_PATH = WORK_DIR / "deeplabv3_resnet50_drive_history.csv"
DRIVE_RESULTS_PATH = WORK_DIR / "deeplabv3_resnet50_drive_results.csv"
DRIVE_PREDICTIONS_DIR = WORK_DIR / "deeplabv3_resnet50_drive_predictions"

In [ ]:
def resolve_processed_dataset_root(data_dir: Path, zip_path: Path, extract_dir: Path) -> Path:
    # Resolve a processed dataset root containing train/val/test folders.
    if zip_path.exists():
        if not extract_dir.exists() or not all((extract_dir / split).exists() for split in ("train", "val", "test")):
            extract_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(extract_dir)

    for root in (extract_dir, data_dir):
        if all((root / split).exists() for split in ("train", "val", "test")):
            return root
        if root.exists():
            for child in root.iterdir():
                if child.is_dir() and all((child / split).exists() for split in ("train", "val", "test")):
                    return child

    raise FileNotFoundError(f"Could not find processed dataset root with train/val/test under {data_dir} or {extract_dir}.")

CHASE_ROOT = resolve_processed_dataset_root(CHASE_DATA_DIR, CHASE_ZIP_PATH, CHASE_EXTRACT_DIR)
DRIVE_ROOT = resolve_processed_dataset_root(DRIVE_DATA_DIR, DRIVE_ZIP_PATH, DRIVE_EXTRACT_DIR)

print("CHASE_ROOT:", CHASE_ROOT)
print("CHASE train files:", len(list((CHASE_ROOT / "train").glob("*.pt"))))
print("DRIVE_ROOT:", DRIVE_ROOT)
print("DRIVE train files:", len(list((DRIVE_ROOT / "train").glob("*.pt"))))

## 3. Dataset and DataLoader

The same dataset class is reused for CHASEDB1 and DRIVE to keep the data pipeline consistent with the SegFormer notebook.

In [ ]:
IMAGENET_MEAN_TENSOR = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3, 1, 1)
IMAGENET_STD_TENSOR = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3, 1, 1)


def prepare_retina_image(image: torch.Tensor) -> torch.Tensor:
    # Normalize RGB image with ImageNet mean/std for ResNet50 backbone.
    image = image.float()
    if image.dim() != 3 or image.shape[0] != 3:
        raise ValueError(f"Expected image [3, H, W], got {tuple(image.shape)}")

    min_value = float(image.min())
    max_value = float(image.max())
    if min_value >= 0.0 and max_value > 1.0:
        image = image / 255.0
        image = (image - IMAGENET_MEAN_TENSOR) / IMAGENET_STD_TENSOR
    elif min_value >= 0.0 and max_value <= 1.0:
        image = (image - IMAGENET_MEAN_TENSOR) / IMAGENET_STD_TENSOR
    return image


class RetinalProcessedDataset(Dataset):
    def __init__(self, root_dir: Path, split: str, img_size: int | None = None):
        self.root_dir = Path(root_dir)
        self.split = split
        self.files = sorted((self.root_dir / split).glob("*.pt"))
        self.img_size = img_size
        if len(self.files) == 0:
            raise FileNotFoundError(f"No .pt files found in {self.root_dir / split}")

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, idx: int):
        sample = torch.load(self.files[idx], map_location="cpu")
        image = sample.get("image")
        mask = sample.get("manual")
        if mask is None:
            mask = sample.get("manual_2")
        if mask is None:
            mask = sample.get("mask")
        if image is None:
            raise KeyError(f"Sample {self.files[idx]} does not contain key 'image'.")
        if mask is None:
            raise KeyError(f"Sample {self.files[idx]} does not contain manual/manual_2/mask.")

        if not torch.is_tensor(image):
            image = torch.tensor(image)
        if image.dim() == 3 and image.shape[0] != 3:
            image = image.permute(2, 0, 1)
        image = prepare_retina_image(image)

        if not torch.is_tensor(mask):
            mask = torch.tensor(mask)
        mask = mask.float()
        if mask.dim() == 2:
            mask = mask.unsqueeze(0)
        elif mask.dim() == 3 and mask.shape[0] != 1:
            if mask.shape[-1] in (1, 3):
                mask = mask.permute(2, 0, 1)
            if mask.shape[0] != 1:
                mask = mask[:1]
        if mask.max() > 1.0:
            mask = mask / 255.0
        mask = (mask > 0.5).float()
        if mask.shape[0] != 1:
            raise ValueError(f"Expected mask [1, H, W], got {tuple(mask.shape)}")

        if self.img_size is not None:
            target_size = (self.img_size, self.img_size)
            if image.shape[-2:] != target_size:
                image = F.interpolate(image.unsqueeze(0), size=target_size, mode="bilinear", align_corners=False).squeeze(0)
            if mask.shape[-2:] != target_size:
                mask = F.interpolate(mask.unsqueeze(0), size=target_size, mode="nearest").squeeze(0)
        return image, mask


def build_loaders(root: Path, img_size: int, batch_size: int):
    train_ds = RetinalProcessedDataset(root, "train", img_size=img_size)
    val_ds = RetinalProcessedDataset(root, "val", img_size=img_size)
    test_ds = RetinalProcessedDataset(root, "test", img_size=img_size)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader


chase_train_ds, chase_val_ds, chase_test_ds, chase_train_loader, chase_val_loader, chase_test_loader = build_loaders(
    CHASE_ROOT, img_size=CHASE_IMG_SIZE, batch_size=CHASE_BATCH_SIZE
)

drive_train_ds, drive_val_ds, drive_test_ds, drive_train_loader, drive_val_loader, drive_test_loader = build_loaders(
    DRIVE_ROOT, img_size=DRIVE_IMG_SIZE, batch_size=DRIVE_BATCH_SIZE
)

print("CHASE train/val/test:", len(chase_train_ds), len(chase_val_ds), len(chase_test_ds))
print("DRIVE train/val/test:", len(drive_train_ds), len(drive_val_ds), len(drive_test_ds))

## 4. DeepLabV3-ResNet50 model

The model uses a **DeepLabV3** segmentation head with a **ResNet50** CNN backbone. The final classifier is replaced to output `1` binary vessel logit channel.

In [ ]:
from src.models.deeplabv3_resnet50 import DeepLabV3ResNet50Binary

def build_deeplabv3_resnet50(pretrained_backbone: bool = True) -> nn.Module:
    return DeepLabV3ResNet50Binary(pretrained_backbone=pretrained_backbone).to(DEVICE)

In [ ]:
# ============================================================
# MODEL SANITY CHECK
# ============================================================

model = build_deeplabv3_resnet50(pretrained_backbone=True)
images, masks = next(iter(drive_train_loader))

with torch.no_grad():
    logits = model(images.to(DEVICE))

expected_shape = (images.size(0), 1, images.size(2), images.size(3))
assert images.ndim == 4 and images.size(1) == 3, f"Expected input [B, 3, H, W], got {tuple(images.shape)}"
assert tuple(masks.shape) == expected_shape, f"Expected masks {expected_shape}, got {tuple(masks.shape)}"
assert tuple(logits.shape) == expected_shape, f"Expected logits {expected_shape}, got {tuple(logits.shape)}"

print("input :", images.shape)
print("masks :", masks.shape)
print("logits:", logits.shape)

## 5. Loss, metrics, early stopping, and training functions

This section follows the SegFormer notebook training style: **BCE + Dice Loss**, `AdamW`, `AMP`, early stopping, and best checkpoint saving.

In [ ]:
import copy

from src.utils.losses import BCEDiceLoss

criterion = BCEDiceLoss(bce_weight=0.4, dice_weight=0.6)

def compute_binary_segmentation_metrics(logits: torch.Tensor, masks: torch.Tensor, threshold: float = 0.5, eps: float = 1e-7):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()
    masks = masks.float()

    preds_flat = preds.view(-1)
    masks_flat = masks.view(-1)

    tp = ((preds_flat == 1) & (masks_flat == 1)).sum().float()
    tn = ((preds_flat == 0) & (masks_flat == 0)).sum().float()
    fp = ((preds_flat == 1) & (masks_flat == 0)).sum().float()
    fn = ((preds_flat == 0) & (masks_flat == 1)).sum().float()

    accuracy = (tp + tn) / (tp + tn + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    iou = tp / (tp + fp + fn + eps)
    dice = 2 * tp / (2 * tp + fp + fn + eps)

    return {
        "accuracy": accuracy.item(),
        "precision": precision.item(),
        "recall": recall.item(),
        "f1": f1.item(),
        "iou": iou.item(),
        "dice": dice.item(),
    }

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device).float()
        if masks.ndim == 3:
            masks = masks.unsqueeze(1)

        optimizer.zero_grad()
        logits = model(images)
        if logits.shape[-2:] != masks.shape[-2:]:
            logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)

@torch.no_grad()
def evaluate(model, loader, criterion, device, threshold: float = 0.5):
    model.eval()
    total_loss = 0.0
    metric_sums = {
        "accuracy": 0.0,
        "precision": 0.0,
        "recall": 0.0,
        "f1": 0.0,
        "iou": 0.0,
        "dice": 0.0,
    }
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device).float()
        if masks.ndim == 3:
            masks = masks.unsqueeze(1)
        logits = model(images)
        if logits.shape[-2:] != masks.shape[-2:]:
            logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
        loss = criterion(logits, masks)
        total_loss += loss.item()
        batch_metrics = compute_binary_segmentation_metrics(logits, masks, threshold=threshold)
        for key in metric_sums:
            metric_sums[key] += batch_metrics[key]
    n = max(len(loader), 1)
    results = {key: value / n for key, value in metric_sums.items()}
    results["loss"] = total_loss / n
    return results

def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs: int = 50,
    patience: int = 10,
    save_path: Path | str = "best_model.pth",
    monitor: str = "dice",
    config: dict | None = None,
    threshold: float = 0.5,
    ):
    best_score = -float("inf")
    best_state = None
    epochs_no_improve = 0
    history = []
    config = config or {}

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = evaluate(model, val_loader, criterion, device, threshold=threshold)
        current_score = val_metrics[monitor]
        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
        history.append(row)
        print(
            f"Epoch [{epoch:03d}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val Dice: {val_metrics['dice']:.4f} | "
            f"Val IoU: {val_metrics['iou']:.4f} | "
            f"Val F1: {val_metrics['f1']:.4f}"
)
        if current_score > best_score:
            best_score = current_score
            best_state = copy.deepcopy(model.state_dict())
            checkpoint = {
                "epoch": epoch,
                "model_state_dict": best_state,
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_dice": val_metrics["dice"],
                "config": config,
            }
            torch.save(checkpoint, save_path)
            epochs_no_improve = 0
            print(f"  -> Save best model: {save_path}")
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch}")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

## 6. Stage 1 — Source training on CHASEDB1

Train DeepLabV3-ResNet50 on CHASEDB1 as the source domain. The saved checkpoint initializes the DRIVE transfer stage.

In [ ]:
source_model = build_deeplabv3_resnet50(pretrained_backbone=True)

source_optimizer = torch.optim.AdamW(
    source_model.parameters(),
    lr=CHASE_LR,
    weight_decay=CHASE_WEIGHT_DECAY,
)

chase_config = {
    "dataset": "CHASEDB1",
    "num_epochs": CHASE_NUM_EPOCHS,
    "lr": CHASE_LR,
    "weight_decay": CHASE_WEIGHT_DECAY,
    "patience": CHASE_PATIENCE,
    "monitor": BEST_METRIC,
    "threshold": THRESHOLD,
}

source_model, chase_history = train_model(
    model=source_model,
    train_loader=chase_train_loader,
    val_loader=chase_val_loader,
    criterion=criterion,
    optimizer=source_optimizer,
    device=DEVICE,
    num_epochs=CHASE_NUM_EPOCHS,
    patience=CHASE_PATIENCE,
    save_path=CHASE_BEST_MODEL_PATH,
    monitor=BEST_METRIC,
    config=chase_config,
    threshold=THRESHOLD,
)

chase_history_df = pd.DataFrame(chase_history)
chase_history_df.to_csv(CHASE_HISTORY_PATH, index=False)
print(f"Saved CHASE training history to {CHASE_HISTORY_PATH}")
chase_history_df.tail()

In [ ]:
best_source_model = build_deeplabv3_resnet50(pretrained_backbone=False)
source_ckpt = torch.load(CHASE_BEST_MODEL_PATH, map_location=DEVICE)
best_source_model.load_state_dict(source_ckpt["model_state_dict"])

chase_val_metrics = evaluate(best_source_model, chase_val_loader, criterion, DEVICE, threshold=THRESHOLD)
chase_test_metrics = evaluate(best_source_model, chase_test_loader, criterion, DEVICE, threshold=THRESHOLD)

best_idx = chase_history_df["val_dice"].idxmax() if BEST_METRIC == "dice" else chase_history_df["val_iou"].idxmax()
chase_best_epoch = int(chase_history_df.loc[best_idx, "epoch"])

chase_results_df = pd.DataFrame([
    {"dataset": "CHASEDB1", "split": "val", "epoch": chase_best_epoch, **chase_val_metrics},
    {"dataset": "CHASEDB1", "split": "test", "epoch": chase_best_epoch, **chase_test_metrics},
])
chase_results_df.to_csv(CHASE_RESULTS_PATH, index=False)
print(f"Saved CHASE results to {CHASE_RESULTS_PATH}")
chase_results_df

## 7. Stage 2 — Transfer learning / fine-tuning on DRIVE

Load the best CHASEDB1 checkpoint, then fine-tune on DRIVE with a smaller learning rate.

In [ ]:
transfer_model = build_deeplabv3_resnet50(pretrained_backbone=False)
source_ckpt = torch.load(CHASE_BEST_MODEL_PATH, map_location=DEVICE)
transfer_model.load_state_dict(source_ckpt["model_state_dict"])
print(f"Loaded source checkpoint from {CHASE_BEST_MODEL_PATH}")

drive_optimizer = torch.optim.AdamW(
    transfer_model.parameters(),
    lr=DRIVE_LR,
    weight_decay=DRIVE_WEIGHT_DECAY,
)

drive_config = {
    "dataset": "DRIVE",
    "num_epochs": DRIVE_NUM_EPOCHS,
    "lr": DRIVE_LR,
    "weight_decay": DRIVE_WEIGHT_DECAY,
    "patience": DRIVE_PATIENCE,
    "monitor": BEST_METRIC,
    "threshold": THRESHOLD,
}

transfer_model, drive_history = train_model(
    model=transfer_model,
    train_loader=drive_train_loader,
    val_loader=drive_val_loader,
    criterion=criterion,
    optimizer=drive_optimizer,
    device=DEVICE,
    num_epochs=DRIVE_NUM_EPOCHS,
    patience=DRIVE_PATIENCE,
    save_path=DRIVE_BEST_MODEL_PATH,
    monitor=BEST_METRIC,
    config=drive_config,
    threshold=THRESHOLD,
)

drive_history_df = pd.DataFrame(drive_history)
drive_history_df.to_csv(DRIVE_HISTORY_PATH, index=False)
print(f"Saved DRIVE training history to {DRIVE_HISTORY_PATH}")
drive_history_df.tail()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(chase_history_df["epoch"], chase_history_df["train_loss"], label="CHASE train loss")
axes[0].plot(chase_history_df["epoch"], chase_history_df["val_loss"], label="CHASE val loss")
axes[0].set_title("Source Training on CHASEDB1")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(drive_history_df["epoch"], drive_history_df["train_loss"], label="DRIVE train loss")
axes[1].plot(drive_history_df["epoch"], drive_history_df["val_loss"], label="DRIVE val loss")
axes[1].set_title("Transfer Learning / Fine-tuning on DRIVE")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 7.1 Threshold tuning on validation set

Tune the inference threshold on DRIVE validation before final test evaluation and visualization.

In [ ]:
best_drive_model_for_tuning = build_deeplabv3_resnet50(pretrained_backbone=False)
drive_ckpt_for_tuning = torch.load(DRIVE_BEST_MODEL_PATH, map_location=DEVICE)
best_drive_model_for_tuning.load_state_dict(drive_ckpt_for_tuning["model_state_dict"])

threshold_grid = [0.40, 0.45, 0.50, 0.55, 0.60]
best_threshold = THRESHOLD
threshold_rows = []

for threshold in threshold_grid:
    metrics = evaluate(best_drive_model_for_tuning, drive_val_loader, criterion, DEVICE, threshold=threshold)
    threshold_rows.append({
        "threshold": threshold,
        "dice": metrics["dice"],
        "iou": metrics["iou"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
    })

threshold_tuning_df = pd.DataFrame(threshold_rows).sort_values("dice", ascending=False).reset_index(drop=True)
best_threshold = float(threshold_tuning_df.loc[0, "threshold"])

print("Validation threshold tuning results:")
print(threshold_tuning_df.to_string(index=False))
print(f"Best threshold: {best_threshold:.2f}")

In [ ]:
best_drive_model = build_deeplabv3_resnet50(pretrained_backbone=False)
drive_ckpt = torch.load(DRIVE_BEST_MODEL_PATH, map_location=DEVICE)
best_drive_model.load_state_dict(drive_ckpt["model_state_dict"])

val_metrics = evaluate(best_drive_model, drive_val_loader, criterion, DEVICE, threshold=best_threshold)
test_metrics = evaluate(best_drive_model, drive_test_loader, criterion, DEVICE, threshold=best_threshold)

best_idx = drive_history_df["val_dice"].idxmax() if BEST_METRIC == "dice" else drive_history_df["val_iou"].idxmax()
drive_best_epoch = int(drive_history_df.loc[best_idx, "epoch"])

drive_results_df = pd.DataFrame([
    {"dataset": "DRIVE", "split": "val", "epoch": drive_best_epoch, **val_metrics},
    {"dataset": "DRIVE", "split": "test", "epoch": drive_best_epoch, **test_metrics},
])
drive_results_df.to_csv(DRIVE_RESULTS_PATH, index=False)
print(f"Saved DRIVE validation/test results to {DRIVE_RESULTS_PATH}")
print(f"Best threshold used for final evaluation: {best_threshold:.2f}")
drive_results_df

## 8. Visualization: Image | Ground Truth | Prediction

The output visualization follows the same style as the SegFormer notebook and matches the expected issue result.

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def denorm_image(img: torch.Tensor) -> np.ndarray:
    if img.dim() == 3 and img.shape[0] == 3:
        img = img.detach().cpu().numpy()
        img = img * IMAGENET_STD[:, None, None] + IMAGENET_MEAN[:, None, None]
        img = np.clip(img, 0.0, 1.0)
        img = np.transpose(img, (1, 2, 0))
        return img
    return img.detach().cpu().numpy()


def show_predictions(model, dataset, save_dir: Path, num_samples: int = 3, threshold: float = THRESHOLD, min_area: int = 10, use_morphology: bool = True):
    model.eval()
    save_dir.mkdir(parents=True, exist_ok=True)
    num_samples = min(num_samples, len(dataset))
    idxs = np.random.choice(len(dataset), size=num_samples, replace=False)
    saved_paths = []
    for idx in idxs:
        sample_id = int(idx)
        image, mask = dataset[idx]
        with torch.no_grad():
            logits = model(image.unsqueeze(0).to(DEVICE))
            prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
            pred = (prob > threshold).astype(np.uint8)
            # Optional morphology: open small noise and keep structure
            if use_morphology:
                kernel = np.ones((3, 3), dtype=np.uint8)
                pred = cv2.morphologyEx(pred, cv2.MORPH_OPEN, kernel)
            # Remove small connected components by contour area
            if min_area and min_area > 0:
                contours, _ = cv2.findContours(pred.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                mask_keep = np.zeros_like(pred, dtype=np.uint8)
                for c in contours:
                    area = cv2.contourArea(c)
                    if area >= float(min_area):
                        cv2.drawContours(mask_keep, [c], -1, color=1, thickness=-1)
                pred = mask_keep.astype(np.float32)
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(denorm_image(image))
        axes[0].set_title("Image")
        axes[0].axis("off")
        axes[1].imshow(mask.squeeze(0).cpu().numpy(), cmap="gray")
        axes[1].set_title("Ground Truth")
        axes[1].axis("off")
        axes[2].imshow(pred, cmap="gray")
        axes[2].set_title("Prediction")
        axes[2].axis("off")
        plt.tight_layout()
        save_path = save_dir / f"prediction_{sample_id:03d}.png"
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        saved_paths.append(save_path)
        plt.show()
        plt.close(fig)
    print(f"Saved {len(saved_paths)} prediction figures to {save_dir}")
    return saved_paths


prediction_paths = show_predictions(best_drive_model, drive_test_ds, save_dir=DRIVE_PREDICTIONS_DIR, num_samples=3, threshold=best_threshold, min_area=10, use_morphology=True)
prediction_paths

## 9. Final summary

This notebook completes the DeepLabV3-ResNet50 CNN model path for the project:

- Model: **DeepLabV3-ResNet50**
- Method: **Transfer Learning**
- Source dataset: **CHASEDB1**
- Target dataset: **DRIVE**
- Loss: **BCEWithLogitsLoss + Dice Loss**
- Metrics: Dice, IoU, Accuracy, Precision, Recall, F1
- Output: checkpoints, CSV histories/results, and Image | Ground Truth | Prediction visualizations

In [ ]:
summary = {
    "model": "DeepLabV3-ResNet50",
    "source_dataset": "CHASEDB1",
    "target_dataset": "DRIVE",
    "training_method": "Transfer Learning: CHASEDB1 source training -> DRIVE fine-tuning",
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "best_metric": BEST_METRIC,
    "threshold": THRESHOLD,
    "chase_best_checkpoint": str(CHASE_BEST_MODEL_PATH),
    "drive_best_checkpoint": str(DRIVE_BEST_MODEL_PATH),
    "chase_history_csv": str(CHASE_HISTORY_PATH),
    "drive_history_csv": str(DRIVE_HISTORY_PATH),
    "drive_results_csv": str(DRIVE_RESULTS_PATH),
    "best_threshold": best_threshold,
    "postprocess_min_area": 10,
    "postprocess_use_morphology": True,
    "drive_predictions_dir": str(DRIVE_PREDICTIONS_DIR),
}
summary_df = pd.DataFrame(list(summary.items()), columns=["Item", "Value"])
summary_df